**# 01 — Setup
 Creates database, schema and logging table for the pipeline**

In [0]:

# DATABASE / SCHEMA
spark.sql("CREATE DATABASE IF NOT EXISTS pakistan_weather")
spark.sql("USE pakistan_weather")

print("Database created: pakistan_weather")
print("All tables will be created under this database")

In [0]:

# PIPELINE EXECUTION LOGS TABLE
spark.sql("""
    CREATE TABLE IF NOT EXISTS pakistan_weather.pipeline_execution_logs (
        log_id          BIGINT GENERATED ALWAYS AS IDENTITY,
        layer           STRING,
        load_type       STRING,
        file_processed  STRING,
        city            STRING,
        start_time      TIMESTAMP,
        end_time        TIMESTAMP,
        status          STRING,
        rows_inserted   BIGINT,
        rows_updated    BIGINT,
        error_message   STRING,
        batch_id        STRING
    )
    USING DELTA
""")

print("Logging table created: pipeline_execution_logs")

In [0]:
spark.sql("""
    ALTER TABLE pakistan_weather.pipeline_execution_logs 
ADD COLUMN rows_aggregated BIGINT;
""")


In [0]:

# VERIFY SETUP
print("Databases available:")
spark.sql("SHOW DATABASES").show()

print("Tables in pakistan_weather:")
spark.sql("SHOW TABLES IN pakistan_weather").show()

print("Setup complete!")

In [0]:
import json

SRC = "dbfs:/Volumes/workspace/default/products/incremental/incremental_2026-10-04.json"   # <- your file
DST = "dbfs:/Volumes/workspace/default/products/incremental/drift_test.json"

data = json.loads(dbutils.fs.head(SRC, 10_000_000))
data[0]["hourly"]["relative_humidity_2m"][0] = "N/A"      # bad value -> quarantine
data[0]["hourly"]["uv_index"] = [1.0] * 24                # new field -> _rescued_data
dbutils.fs.put(DST, json.dumps(data), True)
print("drift file written")